In [7]:
import importlib.util
import sys
from mmengine.config import Config
from mmengine.registry import DATASETS, init_default_scope

# ===== 重新导入你自己的 custom_coco_dataset.py 模块 =====
module_name = 'custom_coco_dataset'
file_path = '/root/MambaVision/object_detection/my_mmdet/datasets/custom_coco_dataset.py'
spec = importlib.util.spec_from_file_location(module_name, file_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)  # ✅ 注册到 DATASETS

# ===== 再次获取已注册的类（现在一定能找到）=====
CustomCocoDataset = DATASETS.get('CustomCocoDataset')
assert CustomCocoDataset is not None, "❌ 注册失败，类仍未获取到！"

# ===== 构建配置 =====
cfg = Config.fromfile('/root/MambaVision/object_detection/configs/mamba_vision/cascade_mask_rcnn_mamba_vision_tiny_3x_coco128.py')
init_default_scope(cfg.default_scope if 'default_scope' in cfg else 'mmdet')
dataset_cfg = cfg.train_dataloader.dataset

# ===== 实例化数据集 =====
dataset = CustomCocoDataset(**dataset_cfg)

# ===== 打印前 4 个样本中 brand_labels 的来源 =====
for i in range(4):
    data = dataset.prepare_data(i)
    sample = data['data_samples']

    print(f"\n===== Sample {i} =====")
    print("img_path:", sample.metainfo.get('img_path', 'N/A'))

    if hasattr(sample.gt_instances, 'brand_labels'):
        print("✅ brand_labels 来自 gt_instances:", sample.gt_instances.brand_labels)
    elif 'brand_id' in sample.metainfo:
        print("⚠️ brand_id 来自 metainfo:", sample.metainfo['brand_id'])
    else:
        print("❌ 没有 brand_id")


KeyError: 'CustomCocoDataset is already registered in dataset at my_mmdet.datasets.custom_coco_dataset'